In [51]:
import pandas as pd
import numpy as np

In [52]:
df = pd.read_csv('/content/spam.csv')

In [53]:
df.sample(5)

,Category,Message
3507,ham,Nite...
2604,ham,Im at arestaurant eating squid! i will be out ...
4109,ham,Pls send me your address sir.
4374,ham,Me fine..absolutly fine
4003,ham,Did you get any gift? This year i didnt get an...


In [54]:
df.describe()

,Category,Message
count,5572,5572
unique,2,5157
top,ham,"Sorry, I'll call later"
freq,4825,30


In [55]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Category  5572 non-null   object
 1   Message   5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [56]:
df['Spam'] = df['Category'].apply(lambda x: 1 if x == 'spam' else 0)
df.head()

,Category,Message,Spam
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


In [58]:
print("--- Absolute Counts ---")
print(df['Spam'].value_counts())

print("\n--- Percentage Distribution ---")
print(df['Spam'].value_counts(normalize=True) * 100)

# highly imbalanced -> 87% normal and only 13% spam

--- Absolute Counts ---
Spam
0    4825
1     747
Name: count, dtype: int64

--- Percentage Distribution ---
Spam
0    86.593683
1    13.406317
Name: proportion, dtype: float64


In [48]:
from sklearn.model_selection import train_test_split

X = df['Message']
y = df['Spam']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [62]:
# convert text to vector
from sklearn.feature_extraction.text import CountVectorizer

In [61]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report, confusion_matrix

lr = LogisticRegression()

vectorizer = CountVectorizer()

X_train_vectorized = vectorizer.fit_transform(X_train)
X_test_vectorized = vectorizer.transform(X_test)

lr.fit(X_train_vectorized, y_train)

lr_pred = lr.predict(X_test_vectorized)
score = accuracy_score(y_test, lr_pred)

# don't rely on accuracy as the data is severly imbalanced, accuracy is only good when data is balanced
print(f"Accuracy: {score:.4f}")

# needed in this case (classification + imbalanced), but not in standard regression (that is, number predictions)
print("--- Classification Report ---")
print(classification_report(y_test, lr_pred))

print("--- Confusion Matrix ---")
print(confusion_matrix(y_test, lr_pred))

Accuracy: 0.9857
--- Classification Report ---
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       966
           1       0.99      0.90      0.94       149

    accuracy                           0.99      1115
   macro avg       0.99      0.95      0.97      1115
weighted avg       0.99      0.99      0.99      1115

--- Confusion Matrix ---
[[965   1]
 [ 15 134]]


In [63]:
# with pipeline, and multiple algorithms test

from sklearn.pipeline import make_pipeline

from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier

models = {
    "Logistic Regression": LogisticRegression(class_weight='balanced', random_state=42),
    "Naive Bayes (Multinomial)": MultinomialNB(),
    "Random Forest": RandomForestClassifier(class_weight='balanced', random_state=42)
}

for name, model in models.items():
  print("=" * 60)
  print(f" TRAINING ALGORITHM: {name} ")
  print("=" * 60)

  pipeline = make_pipeline(CountVectorizer(), model)

  pipeline.fit(X_train, y_train)

  predictions = pipeline.predict(X_test)

  print(classification_report(y_test, predictions))
  print("\n")

 TRAINING ALGORITHM: Logistic Regression 
              precision    recall  f1-score   support

           0       0.99      1.00      0.99       966
           1       0.98      0.93      0.95       149

    accuracy                           0.99      1115
   macro avg       0.98      0.96      0.97      1115
weighted avg       0.99      0.99      0.99      1115



 TRAINING ALGORITHM: Naive Bayes (Multinomial) 
              precision    recall  f1-score   support

           0       0.99      1.00      1.00       966
           1       1.00      0.94      0.97       149

    accuracy                           0.99      1115
   macro avg       1.00      0.97      0.98      1115
weighted avg       0.99      0.99      0.99      1115



 TRAINING ALGORITHM: Random Forest 
              precision    recall  f1-score   support

           0       0.97      1.00      0.99       966
           1       1.00      0.81      0.90       149

    accuracy                           0.97      111

In [67]:
# How to Judge the Metrics?

# Step A: Look at Precision First (The "False Alarm" Metric)
# Step B: Look at Recall Second (The "Catch Rate" Metric)
"""
  • Why? Recall measures what percentage of total spam was successfully blocked.
  • The Verdict: Random Forest has a terrible recall of 0.81, meaning it lets 19% of all spam leak straight into the main inbox. Naive Bayes dominates here with a 0.94 recall (only letting 6% slip through).
"""
# Step C: Verify with the F1-Score
"""
  The F1-score mathematically balances both numbers. Naive Bayes wins definitively with a 0.97 F1-score.
"""

'\n  The F1-score mathematically balances both numbers. Naive Bayes wins definitively with a 0.97 F1-score.\n'